In [0]:
# Celda 1: Definir lista de tablas a ingestar
tablas_a_ingestar = [
    "clientes",
    "detalle_pedido",
    "pedidos",
    "productos"
]

print(f"Total de tablas a ingestar: {len(tablas_a_ingestar)}")
print("Tablas:", tablas_a_ingestar)

In [0]:
# Celda 2: Configurar parámetros de ejecución
MAX_WORKERS = 4  # Número de ejecuciones paralelas
TIMEOUT_POR_TABLA = 900  # 15 minutos por tabla
SCHEMA_DESTINO = "bronze"
NOTEBOOK_INGESTA = "/Workspace/Users/juandavid2931@gmail.com/Python-course-for-data-engineering/02_ELT/01_Ingestas/01_Metadata_Driven/01_motor_de_ingesta"  # Ajusta la ruta

print(f"Workers paralelos: {MAX_WORKERS}")
print(f"Timeout por tabla: {TIMEOUT_POR_TABLA}s")
print(f"Notebook hijo: {NOTEBOOK_INGESTA}")

In [0]:
# Celda 3: Función para ejecutar ingesta de una tabla
from datetime import datetime

def ingestar_tabla(tabla):
    inicio = datetime.utcnow()
    try:
        print(f"▶ Iniciando ingesta: {tabla}")
        
        resultado_str = dbutils.notebook.run(
            NOTEBOOK_INGESTA,
            timeout_seconds=TIMEOUT_POR_TABLA,
            arguments={
                "tabla": tabla
            }
        )
        
        duracion = (datetime.utcnow() - inicio).total_seconds()
        
        return {
            "tabla": tabla,
            "status": "✓ success",
            "duracion_seg": duracion,
            "resultado": resultado_str
        }
        
    except Exception as e:
        duracion = (datetime.utcnow() - inicio).total_seconds()
        return {
            "tabla": tabla,
            "status": "✗ failed",
            "duracion_seg": duracion,
            "error": str(e)
        }

In [0]:
# Celda 4: Ejecutar ingesta en paralelo
from concurrent.futures import ThreadPoolExecutor

print("="*70)
print("INICIANDO INGESTA PARALELA")
print("="*70)

inicio_pipeline = datetime.utcnow()

# Ejecutar todo en paralelo y esperar resultados
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    resultados = list(executor.map(ingestar_tabla, tablas_a_ingestar))

duracion_total = (datetime.utcnow() - inicio_pipeline).total_seconds()

# Mostrar resultados
for r in resultados:
    print(f"{r['status']} {r['tabla']} - {r['duracion_seg']:.1f}s")

print("\n" + "="*70)
print("INGESTA COMPLETADA")
print(f"Duración total: {duracion_total:.2f}s")
print("="*70)